# Omni-Phi: End-to-End Speech-to-Speech Translation Pipeline

This notebook demonstrates the end-to-end training and inference workflow for the **Omni-Phi** S2ST model. Omni-Phi translates spoken English to spoken German directly, bypasses intermediate text decoding, and leverages the frozen EnCodec codes combined with the **microsoft/Phi-4-multimodal-instruct** model's audio capabilities via a speech LoRA adapter.

### Pipeline Stages:
1. **Environment Setup & Path Configuration**
2. **Phase 1: Run Target Tokenization & Preprocessing**
3. **Phase 2 & 3: Load Multimodal Model, Processor, & Dataset**
4. **Phase 4: Run Training Loop using Custom Data Collator**
5. **Phase 5: Run Speech-to-Speech Translation Inference**

## 1. Environment Setup & Path Configuration

First, we import core dependencies and add the project root to `sys.path` so we can import our modules (`dataset`, `model`, `collator`, `inference`, `encoders`, etc.).

In [1]:
import os
import sys
from pathlib import Path

# Resolve directories relative to the current notebook location
notebook_dir = Path(os.getcwd()).resolve()
project_root = notebook_dir.parent.parent
omni_phi_dir = notebook_dir

# Add paths to sys.path
for path in [str(project_root), str(omni_phi_dir)]:
    if path not in sys.path:
        sys.path.append(path)

print(f"Project root  : {project_root}")
print(f"Omni Phi path : {omni_phi_dir}")

Project root  : /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model
Omni Phi path : /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/models/omni_phi


## 2. Phase 1: Preprocessing & Tokenization

Before training, we must extract source waveforms and encode target audios into interleaved EnCodec codebook tokens offset by `+100,000` to fit the Phi-4 vocabulary.

We can trigger Phase 1 preprocessing directly from the notebook. 

> **Note:** For demonstration purposes, we will preprocess a small subset of the `fleurs` dataset. Adjust `--num_samples` as needed for full training.

In [ ]:
# Run Phase 1 Preprocessing script
!python "{project_root}/models/omni_phi/preprocess_omni.py" \
    --dataset fleurs \
    --split train \
    --num_samples 15000 \
    --bandwidth 1.5 \
    --token_offset 100000

## 3. Phase 2 & 3: Load Model, Processor, & Dataset

Now we import our custom dataset class (`OmniPhiDataset`) and core model wrapper (`OmniPhiS2ST`). We will load the underlying multimodal processor, prepare datasets from preprocessed JSONL files, and load our wrapper with the LoRA speech adapter active.

In [2]:
import torch
from transformers import AutoProcessor
from dataset import OmniPhiDataset
from model import OmniPhiS2ST

MODEL_ID = "microsoft/Phi-4-multimodal-instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# 1. Load Processor
print("Loading AutoProcessor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

# 2. Load custom training dataset and evaluation dataset
data_path = omni_phi_dir / "data" / "preprocessed"
train_jsonl = data_path / "train.jsonl"

print(f"Loading dataset from {train_jsonl}...")
train_dataset = OmniPhiDataset(str(train_jsonl), processor, training=True)

# 3. Load Omni-Phi Model Wrapper (with frozen EnCodec and trainable LoRA speech adapter)
print("Loading model wrapper...")
model = OmniPhiS2ST(phi4_model_id=MODEL_ID, device=device)

/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Loading AutoProcessor...


/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading dataset from /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/models/omni_phi/data/preprocessed/train.jsonl...
[OmniPhiDataset] Scanning byte offsets from /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/models/omni_phi/data/preprocessed/train.jsonl to enable low-RAM lazy loading...
[OmniPhiDataset] Scanned 1433 records.
Loading model wrapper...
[OmniPhiS2ST] Loading Phi-4 AutoProcessor...
[OmniPhiS2ST] Loading Phi-4 model...


`torch_dtype` is deprecated! Use `dtype` instead!
/home/zawiatgf/.cache/huggingface/modules/transformers_modules/microsoft/Phi_hyphen_4_hyphen_multimodal_hyphen_instruct/93f923e1a7727d1c4f446756212d9d3e8fcc5d81/speech_conformer_encoder.py:2774: FutureWarning: Please specify CheckpointImpl.NO_REENTRANT as CheckpointImpl.REENTRANT will soon be removed as the default and eventually deprecated.
  lambda i: encoder_checkpoint_wrapper(
Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.35s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


[OmniPhiS2ST] LoRA speech adapter applied. Non-audio layers frozen.
[OmniPhiS2ST] Loading VQGANEncoder (EnCodec)...
[OmniPhiS2ST] VQGANEncoder frozen on cuda.


## 4. Phase 4: Data Collator & Training Loop

With our custom `omni_phi_collate_fn` import, we pad multimodal batch sequences leftwards while masking the prompt region (`labels = -100`), ensuring the model learns exclusively to predict target interleaved EnCodec token sequences.

In [ ]:
from collator import omni_phi_collate_fn
from transformers import Trainer, TrainingArguments

checkpoint_output_dir = omni_phi_dir / "checkpoints"

# Define Hugging Face training arguments
training_args = TrainingArguments(
    output_dir=str(checkpoint_output_dir),
    num_train_epochs=5,
    per_device_train_batch_size=1,  # Reduce memory consumption on small GPUs
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    learning_rate=4e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=1,
    save_strategy="no",             # Set to 'epoch' to save final checkpoint
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    remove_unused_columns=False,    # CRITICAL: Keep custom multimodal inputs
    report_to="none"
)

# Instantiate Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=omni_phi_collate_fn,
    train_dataset=train_dataset,
)

# Start training!
print("Training S2ST model...")
trainer.train()

# Save trained LoRA adapters & processor configurations
trainer.save_model(str(checkpoint_output_dir))
model.processor.save_pretrained(str(checkpoint_output_dir))
print(f"Trained model saved to {checkpoint_output_dir}")

## 5. Phase 5: Inference

Now that the model is loaded and fine-tuned, we can translate spoken English audio to spoken German using both single-pass generation (`translate_speech`) and stream-friendly queue-based generation (`translate_speech_batched`).

In [ ]:
import soundfile as sf
import numpy as np
from inference import translate_speech, translate_speech_batched
from dataset_loader import load_data, save_audio

# ── Load English sample from FLEURS test split ───────────────────────────
print("Loading English sample from FLEURS test split...")
datasets = load_data(
    dataset=["fleurs"],
    lang=["en"],
    split="test",
    num_samples=100,
)
en_ds = datasets.get("en")
if en_ds and len(en_ds) > 0:
    sample = en_ds[0]
    print(f"Loaded sample ID: {sample['id']}")
    print(f"Transcription: {sample['transcription']}")
    print(f"Gender: {sample['gender']}")
    
    # Save the sample audio to 'test_english.wav' for downstream inference steps
    source_wav = "test_english.wav"
    save_audio(sample, source_wav)
    print(f"Saved source audio to: {source_wav}")
else:
    source_wav = "test_english.wav"
    print(f"Warning: FLEURS test split sample not loaded. Expecting '{source_wav}'...")

# ── Option A: Single-Pass S2ST ─────────────────────────────────────────
# This translates an English wav file end-to-end in a single LLM forward-generation pass.
output_wav = "test_translated_german.wav"

if os.path.exists(source_wav):
    print(f"Running single-pass translation on {source_wav}...")
    translate_speech(source_wav_path=source_wav, output_wav_path=output_wav, model=model)
else:
    print(f"Please upload a '{source_wav}' file to run the translation!")

Loading English sample from FLEURS test split...
Loading google/fleurs (en_us) from local storage...
Validating en (audio & uniqueness)...
Final Count: 350 common valid samples.
Loaded sample ID: 1660
Transcription: romanticism had a large element of cultural determinism drawn from writers such as goethe fichte and schlegel
Gender: 1
Saved source audio to: test_english.wav
Running single-pass translation on test_english.wav...
[inference] Translating test_english.wav (sampling rate: 16000 Hz)...
[generate_speech] Generating up to 200 tokens on cuda...


/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


[generate_speech] Generation done. Total ids shape: torch.Size([1, 354])
[generate_speech] Warning: 27/27 codes out-of-range — model may not be sufficiently fine-tuned. Clamping to [0, 1023].
[inference] Done. Output saved to translated_german.wav


In [ ]:
# ── Option B: Queue-Based Batched Inference ────────────────────────────
# Generates audio tokens progressively in small budgets (200 tokens each)
# and queues them together, allowing streaming-style translations.
if os.path.exists(source_wav):
    print(f"Running batched token-queue translation on {source_wav}...")
    audio_data, sr = sf.read(source_wav)
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)
    audio_data = audio_data.astype(np.float32)
    
    # Generate batched waveform
    translated_waveform = translate_speech_batched(model=model, source_audio=audio_data, source_sr=sr)
    
    # Save the output
    batched_output_path = "test_translated_german_batched.wav"
    sf.write(batched_output_path, translated_waveform, samplerate=24000)
    print(f"Batched translation completed successfully! Saved to: {batched_output_path}")